# v6 overlap40 · D-LinkNet-ResNet50 · seed1337
Full五通道、physical batch=2、accumulation=2。只以Val选模并生成Val诊断，**不读取Test**。

In [ ]:
from pathlib import Path
import hashlib, importlib.metadata, importlib.util, json, os, shutil, subprocess, sys

REPO_URL = 'https://github.com/song110585-cpu/lunar-linear.git'
REPO_BRANCH = 'test-new-module'
REPO_DIR = Path('/kaggle/working/lunar-linear')
PROJECT_DIR = REPO_DIR / 'LTL-Net'
OUTPUT_ROOT = Path('/kaggle/working')
CONFIG_FILE = 'v6_overlap40_dlinknet_full_seed1337.json'

required = [('rasterio', 'rasterio'), ('segmentation_models_pytorch', 'segmentation-models-pytorch==0.5.0')]
missing = [package for module, package in required if importlib.util.find_spec(module) is None]
try:
    smp_version = importlib.metadata.version('segmentation-models-pytorch')
except importlib.metadata.PackageNotFoundError:
    smp_version = None
if smp_version != '0.5.0' and 'segmentation-models-pytorch==0.5.0' not in missing:
    missing.append('segmentation-models-pytorch==0.5.0')
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH])
CONFIG_PATH = PROJECT_DIR / 'configs' / CONFIG_FILE
assert CONFIG_PATH.is_file(), f'配置不存在，请先更新仓库: {CONFIG_PATH}'
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
assert config['model'] == 'DLinkNet' and config['seed'] == 1337
assert config['automatic_test_evaluation'] is False
print('Commit:', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

data_paths = [p for p in Path('/kaggle/input').rglob('dataset_v6_random811_overlap40') if p.is_dir() and (p / 'dataset_protocol.json').is_file()]
assert data_paths, '没有找到dataset_v6_random811_overlap40，请先Add Input'
DATA_ROOT = data_paths[0]
for filename, expected in config['expected_metadata_sha256'].items():
    assert sha256(DATA_ROOT / filename) == expected, f'数据元数据不匹配: {filename}'
result_dir = OUTPUT_ROOT / f"result_{config['run_name']}"
assert not result_dir.exists(), f'结果目录已存在，为防覆盖已停止: {result_dir}'
print('数据:', DATA_ROOT)
print('输出:', result_dir)

In [ ]:
command = [
    sys.executable, str(PROJECT_DIR / 'scripts' / 'train_baseline.py'),
    '--model', config['model'], '--encoder', config['encoder'],
    '--data-dir', str(DATA_ROOT), '--output-dir', str(OUTPUT_ROOT),
    '--seed', str(config['seed']), '--epochs', str(config['epochs']),
    '--num-workers', str(config['num_workers']), '--batch-size', str(config['batch_size']),
    '--accum-steps', str(config['accum_steps']), '--run-name', config['run_name'],
    '--skip-test-evaluation',
]
env = os.environ.copy(); env['PYTHONUNBUFFERED'] = '1'
print(' '.join(command), flush=True)
subprocess.check_call(command, cwd=PROJECT_DIR, env=env)
shutil.copy2(CONFIG_PATH, result_dir / 'config.json')

In [ ]:
checkpoint = result_dir / 'best_model.pth'
metrics = json.loads((result_dir / 'metrics.json').read_text(encoding='utf-8'))
assert checkpoint.is_file() and metrics['test'] is None
assert metrics['automatic_test_evaluation'] is False
val_dir = result_dir / 'val_diagnostics'
eval_command = [
    sys.executable, str(PROJECT_DIR / 'scripts' / 'evaluate_segmentation.py'),
    '--model', config['model'], '--encoder', config['encoder'],
    '--data-dir', str(DATA_ROOT), '--checkpoint', str(checkpoint),
    '--output-dir', str(val_dir), '--split', 'val', '--batch-size', '2',
    '--num-workers', str(config['num_workers']), '--channel-mode', 'full',
]
print(' '.join(eval_command), flush=True)
subprocess.check_call(eval_command, cwd=PROJECT_DIR, env=env)
archive = shutil.make_archive(str(result_dir), 'zip', root_dir=result_dir)
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print('下载压缩包:', archive)